# Relatório Mensal de Distância e Emissões

Este notebook carrega o arquivo mestre de viagens aéreas da UFPB, processa as datas e agrupa os totais de distância e emissões por mês, gerando uma tabela e gráficos.

## 1. Importações e Configurações

In [59]:
import pandas as pd
import numpy as np
import altair as alt
import os

# --- Configurações ---
ano = 2025 # Ano dos dados a serem analisados
# orgao = 'UFPB'
orgao = 'UFCG'
# Caminho para o arquivo de entrada 
pasta_dados = f'dadosViagens/dados_viagens{ano}'
arquivo_entrada = os.path.join(pasta_dados, f'df_master_ufpb_aereo_{ano}.csv')

# Colunas que serão usadas do arquivo de entrada
COL_ID_VIAGEM = 'Identificador do processo de viagem'
COL_DATA = 'Período - Data de início' # Coluna com a data de início da viagem
COL_DISTANCIA = 'Distância (GCD)'       # Coluna de distância total da viagem
COL_EMISSOES = 'Emissões (KgCO2eq)'    # Coluna de emissões totais da viagem


## 2. Carregar e Preparar Dados

In [60]:
print(f"🔄 Carregando dados de: '{arquivo_entrada}'...")
try:
    df = pd.read_csv(arquivo_entrada)
    print(f"✅ Dados carregados com sucesso ({len(df)} viagens).")
    
    # --- Verificação de Colunas --- 
    colunas_necessarias = [COL_ID_VIAGEM, COL_DATA, COL_DISTANCIA, COL_EMISSOES]
    colunas_faltantes = [col for col in colunas_necessarias if col not in df.columns]
    if colunas_faltantes:
        print(f"❌ ERRO: Colunas essenciais não encontradas: {colunas_faltantes}")
        # Cria DF vazio para parar a execução das próximas células
        df = pd.DataFrame()
    else:
        print("✅ Colunas necessárias (Data, Distância, Emissões) encontradas.")

except FileNotFoundError:
    print(f"❌ ERRO: Arquivo '{arquivo_entrada}' não encontrado.")
    df = pd.DataFrame() 
except Exception as e:
    print(f"❌ Ocorreu um erro ao carregar os dados: {e}")
    df = pd.DataFrame() 

# --- Processamento de Datas --- 
if not df.empty:
    print("🔄 Processando datas...")
    # Converte a coluna de data para o formato datetime
    # Assumindo formato DD/MM/YYYY (ajuste 'format' se for diferente)
    df['Data_Viagem'] = pd.to_datetime(df[COL_DATA], format='%d/%m/%Y', errors='coerce')
    
    # Remove linhas onde a data não pôde ser convertida
    linhas_antes = len(df)
    df.dropna(subset=['Data_Viagem'], inplace=True)
    linhas_removidas = linhas_antes - len(df)
    if linhas_removidas > 0:
        print(f"   - {linhas_removidas} viagens removidas por data inválida.")
        
    # Cria colunas para agrupar por mês
    df['Mes_Ano'] = df['Data_Viagem'].dt.strftime('%Y-%m') # Formato '2023-01' para rótulos
    df['Mes_Num'] = df['Data_Viagem'].dt.month # Número 1-12 para ordenação
    
    print("✅ Datas processadas e colunas de Mês criadas.")
    print(df[['Data_Viagem', 'Mes_Ano', 'Mes_Num']].head())
else:
    print("⚠️ DataFrame vazio, processamento de datas pulado.")

🔄 Carregando dados de: 'dadosViagens/dados_viagens2025\df_master_ufpb_aereo_2025.csv'...
✅ Dados carregados com sucesso (196 viagens).
✅ Colunas necessárias (Data, Distância, Emissões) encontradas.
🔄 Processando datas...
✅ Datas processadas e colunas de Mês criadas.
  Data_Viagem  Mes_Ano  Mes_Num
0  2025-03-16  2025-03        3
1  2025-03-16  2025-03        3
2  2025-03-09  2025-03        3
3  2025-03-11  2025-03        3
4  2025-03-16  2025-03        3


## 3. Agrupar por Mês e Gerar Relatório

In [61]:
if not df.empty:
    print(f"🔄 Agrupando totais por Mês/Ano para {ano}...")
    
    # Agrupa por Mês/Ano e soma as colunas de interesse
    df_mensal = df.groupby(['Mes_Ano', 'Mes_Num']).agg(
        Total_Distancia_Km = (COL_DISTANCIA, 'sum'),
        Total_Emissoes_KgCO2eq = (COL_EMISSOES, 'sum'),
        Total_Viagens = (COL_ID_VIAGEM, 'count')
    ).reset_index()
    
    # Ordena pelo número do mês
    df_mensal.sort_values('Mes_Num', inplace=True)
    
    print("✅ Agrupamento concluído.")
    
    # --- Exibir Tabela de Resultados --- 
    print("\n--- Relatório Mensal: Totais de Viagens Aéreas (UFPB {ano}) ---")
    colunas_exibir = ['Mes_Ano', 'Total_Viagens', 'Total_Distancia_Km', 'Total_Emissoes_KgCO2eq']
    print(df_mensal[colunas_exibir].to_string(index=False, formatters={
        'Total_Distancia_Km': '{:,.0f} km'.format,
        'Total_Emissoes_KgCO2eq': '{:,.0f} kg'.format
    }))
    
    # --- Salvar Relatório CSV --- 
    try:
        nome_arquivo_saida = f"relatorio_mensal_ufpb_aereo_{ano}.csv"
        caminho_saida = os.path.join(pasta_dados, nome_arquivo_saida)
        df_mensal.round(2).to_csv(caminho_saida, index=False)
        print(f"\n✅ Relatório mensal salvo em: '{caminho_saida}'")
    except Exception as e_save:
        print(f"❌ Erro ao salvar relatório mensal: {e_save}")
        
else:
    print("⚠️ DataFrame vazio, agrupamento pulado.")
    df_mensal = pd.DataFrame() # Cria DF vazio


🔄 Agrupando totais por Mês/Ano para 2025...
✅ Agrupamento concluído.

--- Relatório Mensal: Totais de Viagens Aéreas (UFPB {ano}) ---
Mes_Ano  Total_Viagens Total_Distancia_Km Total_Emissoes_KgCO2eq
2025-01              2           5,670 km               1,037 kg
2025-02              6          20,805 km               3,910 kg
2025-03             19          71,648 km              13,641 kg
2025-04             24          86,900 km              16,585 kg
2025-05             37         160,344 km              30,892 kg
2025-06             20          79,436 km              15,348 kg
2025-07             20          72,509 km              14,041 kg
2025-08             34         112,002 km              21,627 kg
2025-09             34         121,924 km              23,361 kg

✅ Relatório mensal salvo em: 'dadosViagens/dados_viagens2025\relatorio_mensal_ufpb_aereo_2025.csv'


## 4. Visualização (Gráficos Mensais)

In [62]:
# --- CÉLULA 6 (CORRIGIDA): Visualização (Gráficos Mensais) ---

if not df_mensal.empty:
    print("🔄 Criando gráficos mensais...")
    
    # Gráfico base (para tooltips e ordenação)
    base = alt.Chart(df_mensal).encode(
        # Ordena o eixo X pelo número do mês, mas exibe o nome do mês/ano
        x=alt.X('Mes_Ano', sort=alt.EncodingSortField(field="Mes_Num", op="min", order='ascending'), axis=alt.Axis(title='Mês')), 
        tooltip=['Mes_Ano',
                 alt.Tooltip('Total_Viagens', title='Nº de Viagens'),
                 alt.Tooltip('Total_Distancia_Km', title='Distância (km)', format=',.0f'),
                 alt.Tooltip('Total_Emissoes_KgCO2eq', title='Emissões (kg)', format=',.0f')
                ]
    )
    
    # Gráfico 1: Emissões por Mês
    chart_emissoes = base.mark_bar(color='#d9534f').encode(
        y=alt.Y('Total_Emissoes_KgCO2eq', axis=alt.Axis(title='Total Emissões (KgCO2eq)'))
    ).properties(
        title='Emissões Totais por Mês',
        width=700  # <<<--- LARGURA ADICIONADA
    )#.interactive() # Adiciona interatividade (zoom/pan)
    
    # Gráfico 2: Distância por Mês
    chart_distancia = base.mark_bar(color='#4682b4').encode(
        y=alt.Y('Total_Distancia_Km', axis=alt.Axis(title='Total Distância (Km)'))
    ).properties(
        title='Distância Total por Mês',
        width=700  # <<<--- LARGURA ADICIONADA
    )#.interactive()
    
    # Gráfico 3: Número de Viagens por Mês
    chart_viagens = base.mark_line(color='#5cb85c', point=True).encode(
        y=alt.Y('Total_Viagens', axis=alt.Axis(title='Nº de Viagens'))
    ).properties(
        title='Número de Viagens por Mês',
        width=700  # <<<--- LARGURA ADICIONADA
    )#.interactive()

    # Combina os gráficos
    dashboard_mensal = alt.vconcat(chart_emissoes, chart_distancia, chart_viagens).properties(
        title=f"Análise Mensal de Viagens Aéreas - {orgao} {ano}"
    ).resolve_scale(x='shared') # Compartilha o eixo X
    

    # Salvar o dashboard como HTML
    try:
        arquivo_dashboard_html = os.path.join(pasta_dados, f'dashboard_mensal_{orgao}_aereo_{ano}.html')
        dashboard_mensal.save(arquivo_dashboard_html)
        print(f"\n✅ Dashboard mensal salvo como HTML em: '{arquivo_dashboard_html}'")
        print("   (Abra este arquivo .html no seu navegador para ver o gráfico.)")

    except Exception as e_save_chart:
        print(f"❌ Erro ao salvar dashboard mensal como HTML: {e_save_chart}")
        
    # --- EXIBIR O GRÁFICO NO NOTEBOOK ---
    print("\nExibindo dashboard no notebook:")
    dashboard_mensal # <<<--- LINHA DESCOMENTADA

else:
    print("⚠️ DataFrame mensal vazio, gráficos pulados.")

🔄 Criando gráficos mensais...

✅ Dashboard mensal salvo como HTML em: 'dadosViagens/dados_viagens2025\dashboard_mensal_UFCG_aereo_2025.html'
   (Abra este arquivo .html no seu navegador para ver o gráfico.)

Exibindo dashboard no notebook:


In [63]:
chart_emissoes

alt.Chart(...)

In [66]:
chart_distancia

alt.Chart(...)

In [65]:
chart_viagens

alt.Chart(...)